# Analytics Snippets

Small patterns you can copy into the workbook if you get stuck. Replace the column names with the ones used in the task cell. If you already know the pattern, work straight in the participant notebook.

In [ ]:
import pandas as pd

df = pd.read_csv('wind_turbine_telemetry.csv', parse_dates=['Timestamp'])
print(df.head())
print(df.info())
print(df.isna().sum())

In [ ]:
clean_df = df.copy()
clean_df = clean_df.sort_values(['Turbine_ID', 'Timestamp'])
clean_df = clean_df.ffill()

clean_df['Vibration_per_RPM'] = clean_df['Vibration_mm_s'] / clean_df['RPM'].replace(0, pd.NA)
clean_df['Vibration_per_RPM_Rolling'] = (
    clean_df.groupby('Turbine_ID')['Vibration_per_RPM']
    .transform(lambda series: series.rolling(window=12, min_periods=1).mean())
)

In [ ]:
import plotly.express as px

fig = px.scatter(
    clean_df,
    x='Bearing_Temperature_C',
    y='Vibration_mm_s',
    color='Turbine_ID',
    size='RPM',
    hover_data=['Wind_Speed_mps', 'Wind_Gust_mps', 'Turbulence_Intensity'],
)
fig.show()

fig = px.line(clean_df, x='Timestamp', y='Bearing_Temperature_C', color='Turbine_ID')
fig.update_traces(connectgaps=False)
fig.show()

In [ ]:
summary = clean_df.groupby('Turbine_ID').agg({
    'Power_kW': 'mean',
    'Vibration_mm_s': 'max',
    'Maintenance_Flag': 'last',
})
print(summary)

# Use the summary table with one chart to explain the maintenance choice.
# Keep the story tied to the sensor readings, not just the chart shape.